# SpecDrift-Bench — pilot sur Kaggle (gratuit)

Avant de lancer : **Settings → Accelerator : GPU T4 x2** et **Internet : On** (Kaggle demande de vérifier ton numéro de téléphone pour activer Internet).

Ce notebook :
1. installe le benchmark et vLLM ;
2. sert un modèle de code open-weight sur les 2 GPU T4 (aucune clé API, aucun quota de tokens) ;
3. lance un test de 2 tours, puis le pilote ;
4. met les résultats dans `/kaggle/working/runs.zip` à télécharger.

In [ ]:
!git clone -q https://github.com/roanenicki/specdrift-bench.git
%cd specdrift-bench
!pip install -q mini-swe-agent pyyaml pytest vllm
!pip uninstall -y -q torchaudio  # le torchaudio de Kaggle fait planter vLLM (versions CUDA différentes)
!python -m pytest -q tests  # doit afficher: 36 passed

## 1. Démarrer le serveur du modèle

Modèle par défaut : Qwen2.5-Coder-14B-Instruct en version AWQ (quantifiée, tient sur 2×T4).
Le premier démarrage télécharge environ 10 Go et peut prendre 10 minutes.

In [ ]:
import subprocess, time, urllib.request, os
MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--quantization", "awq", "--dtype", "float16", "--tensor-parallel-size", "2",
     "--max-model-len", "32768", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5); print("serveur prêt"); break
    except Exception:
        time.sleep(10)
else:
    print("le serveur ne démarre pas : regarde /kaggle/working/vllm.log")
os.environ["MSWEA_SILENT_STARTUP"] = "1"
os.environ["MSWEA_COST_TRACKING"] = "ignore_errors"

## 2. Test rapide : 2 tours, condition « continuous »

In [ ]:
!python -m specdrift.runner --task tasks/t01_invlite \
    --model hosted_vllm/Qwen/Qwen2.5-Coder-14B-Instruct-AWQ --api-base http://localhost:8000/v1 \
    --condition continuous --n-turns 2 --out /kaggle/working/runs_smoke

## 3. Pilote : 3 conditions × 2 ordres (lignes du plan de Williams)

Les 6 sessions tournent en parallèle. Pour ne pas dépendre de ton navigateur, lance tout le notebook avec
**Save Version → Save & Run All (Commit)** : Kaggle l'exécute en arrière-plan (12 h maximum)
et les résultats apparaissent dans l'onglet **Output** de la version.

In [ ]:
import subprocess
# Les sessions tournent 2 par 2 : au-delà, le cache des GPU T4 sature et les requêtes expirent.
# Chaque session écrit dans son propre dossier et son propre fichier log.
M = "hosted_vllm/Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
procs = []
for row in [0, 1]:
    for cond in ["continuous", "recall", "fresh"]:
        log = open(f"/kaggle/working/pilot_row{row}_{cond}.log", "w")
        procs.append(subprocess.Popen(
            ["python", "-m", "specdrift.runner", "--task", "tasks/t01_invlite", "--model", M,
             "--api-base", "http://localhost:8000/v1", "--condition", cond,
             "--order-row", str(row), "--out", "/kaggle/working/runs"],
            stdout=log, stderr=subprocess.STDOUT))
        print(f"lancé : row {row} / {cond}", flush=True)
    if len(procs) % 2 == 0:  # 2 sessions at a time: the KV cache of 2xT4 holds ~2.4 contexts of 32k tokens
        for p in procs[-2:]:
            p.wait()
for p in procs:
    p.wait()
print("pilote terminé")

## 4. Résumé et export

In [ ]:
import json, glob
for f in sorted(glob.glob("/kaggle/working/runs/**/results.json", recursive=True)):
    r = json.load(open(f))
    print(f.split("runs/")[1].rsplit("/", 1)[0], "| F par tour:", [t["F"] for t in r["turns"]],
          "| fonctionnel:", r["summary"]["functional_success_rate"])
!cd /kaggle/working && zip -qr runs.zip runs runs_smoke vllm.log && ls -lh runs.zip